In [ ]:
!pip install scikit-image
!pip install gradio

In [ ]:
from google.colab import files  # 如果你用的是 Jupyter 可略過這行

uploaded = files.upload()  # 執行後會跳出上傳視窗


Saving G_epoch200.pth to G_epoch200.pth


In [ ]:
!pip install torch torchvision matplotlib scikit-image tqdm --quiet
import os, random, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from torchvision.models import vgg16
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('✅ 使用裝置:', device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.5 MB/s eta 0:00:00
✅ 使用裝置: cpu


In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        def down(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return nn.Sequential(*layers)

        def up(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.BatchNorm2d(out_c),
                nn.ReLU()
            )

        self.enc1 = down(1, 64, False)
        self.enc2 = down(64, 128)
        self.enc3 = down(128, 256)
        self.enc4 = down(256, 512)
        self.enc5 = down(512, 512)
        self.dec1 = up(512, 512)
        self.dec2 = up(1024, 256)
        self.dec3 = up(512, 128)
        self.dec4 = up(256, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh())

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        d1 = self.dec1(e5)
        d2 = self.dec2(torch.cat([d1, e4], 1))
        d3 = self.dec3(torch.cat([d2, e3], 1))
        d4 = self.dec4(torch.cat([d3, e2], 1))
        return self.final(torch.cat([d4, e1], 1))

class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm: layers.append(nn.InstanceNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return layers
        self.model = nn.Sequential(
            *block(4, 64, False),
            *block(64, 128),
            *block(128, 256),
            *block(256, 512),
            nn.ZeroPad2d((1,0,1,0)),
            nn.Conv2d(512, 1, 4, padding=1)
        )

    def forward(self, a, b):
        return self.model(torch.cat([a, b], 1))

G = UNetGenerator().to(device)
D = PatchDiscriminator().to(device)


In [ ]:
import torch.nn as nn
import torch

class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        def down(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return nn.Sequential(*layers)

        def up(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.BatchNorm2d(out_c),
                nn.ReLU()
            )

        self.enc1 = down(1, 64, norm=False)
        self.enc2 = down(64, 128)
        self.enc3 = down(128, 256)
        self.enc4 = down(256, 512)
        self.enc5 = down(512, 512)

        self.dec1 = up(512, 512)
        self.dec2 = up(1024, 256)
        self.dec3 = up(512, 128)
        self.dec4 = up(256, 64)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        d1 = self.dec1(e5)
        d2 = self.dec2(torch.cat([d1, e4], dim=1))
        d3 = self.dec3(torch.cat([d2, e3], dim=1))
        d4 = self.dec4(torch.cat([d3, e2], dim=1))
        out = self.final(torch.cat([d4, e1], dim=1))
        return out


In [ ]:
G = UNetGenerator().to(device)
G.load_state_dict(torch.load("/content/G_epoch200.pth", map_location=device))
G.eval()


UNetGenerator(
  (enc1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.2)
  )
  (enc2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
  )
  (enc3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
  )
  (enc4): Sequential(
    (0): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
  )
  (enc5): Sequential(
    (0): Conv2d(512, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, a

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr

# ✅ 定義 UNetGenerator 架構
class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        def down(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return nn.Sequential(*layers)

        def up(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.BatchNorm2d(out_c),
                nn.ReLU()
            )

        self.enc1 = down(1, 64, False)
        self.enc2 = down(64, 128)
        self.enc3 = down(128, 256)
        self.enc4 = down(256, 512)
        self.enc5 = down(512, 512)
        self.dec1 = up(512, 512)
        self.dec2 = up(1024, 256)
        self.dec3 = up(512, 128)
        self.dec4 = up(256, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh())

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        d1 = self.dec1(e5)
        d2 = self.dec2(torch.cat([d1, e4], 1))
        d3 = self.dec3(torch.cat([d2, e3], 1))
        d4 = self.dec4(torch.cat([d3, e2], 1))
        return self.final(torch.cat([d4, e1], 1))

# ✅ 載入模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetGenerator().to(device)
model.load_state_dict(torch.load("/content/G_epoch200.pth", map_location=device))
model.eval()

# ✅ 上色並評分
def colorize_and_score(bw_img, color_img):
    # transform
    transform_bw = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.Grayscale(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    transform_color = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])

    input_tensor = transform_bw(bw_img).unsqueeze(0).to(device)  # [1, 1, H, W]
    gt_tensor = transform_color(color_img)  # [3, H, W]

    # 上色
    with torch.no_grad():
        output = model(input_tensor)[0].cpu()  # [3, H, W]
        output = (output + 1) / 2
        output_clamped = output.clamp(0,1)
        pred_img = transforms.ToPILImage()(output_clamped)

    # 計算 SSIM/PSNR
    pred_np = np.array(pred_img.resize((256, 256))).astype(np.float32)
    gt_np = np.array(transforms.ToPILImage()(gt_tensor).resize((256, 256))).astype(np.float32)

    ssim = compare_ssim(pred_np, gt_np, channel_axis=2, data_range=255)
    psnr = compare_psnr(gt_np, pred_np, data_range=255)

    score_text = f"✅ SSIM：{ssim:.4f}\n✅ PSNR：{psnr:.2f} dB"

    return pred_img, score_text

# ✅ Gradio 介面
demo = gr.Interface(
    fn=colorize_and_score,
    inputs=[
        gr.Image(type="pil", label="上傳黑白漫畫圖"),
        gr.Image(type="pil", label="原始彩色圖（用來比較）")
    ],
    outputs=[
        gr.Image(type="pil", label="自動上色結果"),
        gr.Textbox(label="📊 SSIM / PSNR 評分結果")
    ],
    title="🎨 黑白漫畫自動上色系統 + 評分",
    description="請上傳黑白圖與原彩圖，上色後會自動計算 SSIM / PSNR 分數"
)

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d7ce161ab4ee57578.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
